## Multipatch stiffness assembly

Builds on [05_multipatch](05_multipatch.ipynb): once a `PatchAssembly` has shared control points, merged interfaces and consistent DOF numbering, `PatchIntegrator.assemble()` turns it into a single global stiffness matrix.

This notebook builds the same rectangular domain in two ways -- as a single patch, and as two patches sharing the middle edge -- and checks that splitting a domain into conforming patches does not change the assembled physics. The geometry (a 6x1 rectangle, degree 2, two elements) is the one already validated against the legacy Fortran solver in `tests/future/test_stiffness.py::test_integration_2_elements_C0`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from yeti_iga.future.bspline import (BSpline, BSplineSurface, ControlPointManager,
    Patch, GlobalDOFManager, PatchDOFManager, PatchAssembly, IGABasis1D,
    PatchIntegrator, MaterialProperties)
from yeti_iga.future.plotting import plot_patches_2d

### The domain as a single patch

15 control points (5x3, u-fastest), degree 2 in both directions. The `u` knot vector `[0, 0, 0, 0.5, 0.5, 1, 1, 1]` has multiplicity 2 at the interior knot 0.5: that's C0 continuity, i.e. two elements glued without extra smoothness -- geometrically the same as two separate patches glued at `x=3`.

In [ ]:
mgr_single = ControlPointManager(dim=2)
for y in (0.0, 0.5, 1.0):
    for x in (0.0, 1.5, 3.0, 4.5, 6.0):
        mgr_single.add_point([x, y])

su_single = BSpline(2, np.array([0., 0., 0., 0.5, 0.5, 1., 1., 1.]))
sv_single = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
mapping_single = list(range(15))

dof_manager_single = GlobalDOFManager([2] * mgr_single.n_points)
patch_dof_manager_single = PatchDOFManager(2, mapping_single, dof_manager_single)
patch_single = Patch(BSplineSurface(su_single, sv_single), mgr_single,
                      mapping_single, [5, 3], patch_dof_manager_single)

basis_u_single = IGABasis1D.build(su_single, 3)
basis_v_single = IGABasis1D.build(sv_single, 3)
material = MaterialProperties(210000, 0.3)

K_single = PatchIntegrator(patch_single, basis_u_single, basis_v_single, material).integrate()
print('K_single shape:', K_single.shape)

### The same domain as two patches sharing the middle edge

Each patch covers one element. The column of control points at `x=3` (global ids 2, 7, 12) is shared *by construction*: both patches reference the same ids in the same `ControlPointManager`.

In [ ]:
mgr = ControlPointManager(dim=2)
for y in (0.0, 0.5, 1.0):
    for x in (0.0, 1.5, 3.0, 4.5, 6.0):
        mgr.add_point([x, y])

dofs_per_control_point = [2] * mgr.n_points
global_dof_manager = GlobalDOFManager(dofs_per_control_point)

# Left patch: columns iu=0,1,2 -> ids 0,1,2 / 5,6,7 / 10,11,12
su_left = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
sv_left = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
mapping_left = [0, 1, 2, 5, 6, 7, 10, 11, 12]
dof_manager_left = PatchDOFManager(2, mapping_left, global_dof_manager)
patch_left = Patch(BSplineSurface(su_left, sv_left), mgr, mapping_left, [3, 3],
                    dof_manager_left)

# Right patch: columns iu=2,3,4 -> ids 2,3,4 / 7,8,9 / 12,13,14 (column 2 shared)
su_right = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
sv_right = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
mapping_right = [2, 3, 4, 7, 8, 9, 12, 13, 14]
dof_manager_right = PatchDOFManager(2, mapping_right, global_dof_manager)
patch_right = Patch(BSplineSurface(su_right, sv_right), mgr, mapping_right, [3, 3],
                     dof_manager_right)

assembly = PatchAssembly()
assembly.add_patch(patch_left)
assembly.add_patch(patch_right)
assembly.detect_shared_control_points()

shared_map = assembly.get_shared_control_points_map()
print('Shared control points:', shared_map)

In [ ]:
plot_patches_2d(assembly, show_control_points=True, show_control_point_indices=True,
                title='Two patches sharing the x=3 edge')

### Assembling the global stiffness matrix

`PatchIntegrator.assemble()` takes the assembly and one `MaterialProperties` per patch (in `add_patch()` order). Here both patches get the *same* material, since we want to compare against the single-patch reference above.

In [ ]:
materials = [material, material]
K_multi = PatchIntegrator.assemble(assembly, materials)
print('K_multi shape:', K_multi.shape)

### Splitting the domain must not change the physics

`K_single` and `K_multi` were built from the same 15 control points, in the same order -- the only difference is whether the domain is one patch or two glued patches. They must be numerically identical: this is exactly the self-consistency check used in `tests/future/test_multipatch_stiffness.py`.

In [ ]:
diff = np.abs(K_single.toarray() - K_multi.toarray())
print('max abs difference:', diff.max())
assert np.allclose(K_single.toarray(), K_multi.toarray(), rtol=1.e-10, atol=1.e-8)
print('K_single == K_multi: OK')

### Sparsity pattern

The shared edge control points (2, 7, 12) are exactly where both patches' contributions overlap and get summed -- no special-casing was needed, it falls out of merging every patch's triplets before the single `setFromTriplets()` call inside `assemble()`.

In [ ]:
plt.figure(figsize=(4, 4))
plt.spy(K_multi, markersize=4)
plt.title('Sparsity pattern of the assembled global stiffness matrix')
plt.show()

### Materials can differ per patch

Unlike the comparison above, `materials` doesn't have to repeat the same `MaterialProperties` for every patch -- a stiffened panel, for example, commonly assigns a different material (or thickness) per patch (see the `OPT_stiffPanel` example). Stiffening the right patch increases the stiffness at the shared edge, since that dof now collects a larger contribution from the right patch.

In [ ]:
materials_mixed = [material, MaterialProperties(2 * 210000, 0.3)]
K_mixed = PatchIntegrator.assemble(assembly, materials_mixed)

# x-dof of the shared control point with id 2 (middle of the shared edge, y=0)
shared_dof = global_dof_manager.get_dof_indices(2)[0]
print('diagonal term at the shared dof, uniform materials:', K_multi[shared_dof, shared_dof])
print('diagonal term at the shared dof, right patch stiffened:', K_mixed[shared_dof, shared_dof])
assert K_mixed[shared_dof, shared_dof] > K_multi[shared_dof, shared_dof]